In [0]:
# Cell 1: Set up catalog, database, and paths using Unity Catalog
import re

# Use your username to make paths unique to your team
username = spark.sql("SELECT current_user()").first()[0]
# Clean username to be usable as a folder name
safe_user = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])

# Define catalog and schema (database) names
CATALOG  = "workspace"
SCHEMA   = "theateriq_team12"
RAW_PATH = f"/tmp/{safe_user}/theateriq/raw/movielens"

print(f"Username:  {username}")
print(f"Catalog:   {CATALOG}")
print(f"Schema:    {SCHEMA}")
print(f"Raw path:  {RAW_PATH}")

# Create schema if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("\nCatalog and schema ready!")

In [0]:
# Cell 2: Download MovieLens 100K directly to Databricks
import subprocess
import os

# Download zip file
subprocess.run([
    "wget", "-q", "-O", "/tmp/ml-100k.zip",
    "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
], check=True)
print("Download complete")

# Unzip it
subprocess.run([
    "unzip", "-q", "-o", "/tmp/ml-100k.zip", "-d", "/tmp/"
], check=True)
print("Unzip complete")

# Show what files came out
files = os.listdir("/tmp/ml-100k/")
print(f"\nFiles in dataset ({len(files)} total):")
for f in sorted(files):
    print(f"  {f}")

In [0]:
# Cell 3: Load ratings into Bronze Delta table
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType

# First copy the file to a Volumes path that Serverless can access
import subprocess
subprocess.run(["mkdir", "-p", f"/tmp/halve566/theateriq/raw/movielens"], check=True)

# Use spark to read directly from the downloaded location
# by first writing it to a UC Volume accessible path
with open("/tmp/ml-100k/u.data", "r") as f:
    lines = f.readlines()

# Parse manually and create a DataFrame
data = []
for line in lines:
    parts = line.strip().split("\t")
    if len(parts) == 4:
        data.append((int(parts[0]), int(parts[1]), float(parts[2]), int(parts[3])))

# Create Spark DataFrame from parsed data
ratings_df = spark.createDataFrame(data, ["userId", "movieId", "rating", "timestamp"])

# Add ingestion metadata
ratings_bronze = ratings_df \
    .withColumn("ingested_at", F.current_timestamp()) \
    .withColumn("source", F.lit("movielens_100k"))

# Write to Unity Catalog as a Delta table
ratings_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.ratings_bronze")

print(f"Ratings loaded: {ratings_bronze.count():,} rows")
display(ratings_bronze.limit(5))

In [0]:
# Cell 4: Load movies into Bronze Delta table
# u.item is pipe separated, we only need first 3 columns
# Format: movieId | title | releaseDate (and many more columns we don't need yet)

data = []
with open("/tmp/ml-100k/u.item", "r", encoding="latin-1") as f:
    for line in f:
        parts = line.strip().split("|")
        if len(parts) >= 3:
            try:
                movie_id    = int(parts[0])
                title       = parts[1]
                release_date = parts[2]
                data.append((movie_id, title, release_date))
            except:
                pass

# Create Spark DataFrame
movies_df = spark.createDataFrame(data, ["movieId", "title", "releaseDate"])

# Add ingestion metadata
movies_bronze = movies_df \
    .withColumn("ingested_at", F.current_timestamp()) \
    .withColumn("source", F.lit("movielens_100k"))

# Write to Unity Catalog as a Delta table
movies_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.movies_bronze")

print(f"Movies loaded: {movies_bronze.count():,} rows")
display(movies_bronze.limit(5))